# CPE 342 Term Project — Task 2: Player Segment Classification (Multi-Class)
**Course:** CPE 342 Machine Learning (Semester 1/2026) — KMUTT  
**System Component:** Karena Thailand Player Intelligence System — Behavioral Marketing & Personalization Module  
**Task Category:** Supervised Multi-Class Classification (`segment`: `0` = Casual, `1` = Grinder, `2` = Social, `3` = Whale)  
**Primary Evaluation Metric:** **Macro $F_1$ Score** ($\frac{1}{4}\sum_{c=0}^3 F_{1,c}$)  
**Competition Rule Compliance:**
* **Data Usage Policy (Rule 2):** Strictly fits all frequency maps, encoders, and models on `train.csv` only (`test.csv` is never used for training, pseudo-labeling, or encoder fitting). Supports combining Train + Validation sets for final refit after validation.
* **Model Development (Rule 3):** Trained **from scratch** on provided data only (`LightGBM` + `XGBoost`). No AutoML frameworks; hyperparameter tuning via `Optuna` is supported.
* **Portable Execution:** Supports **Google Colab GPU**, **Kaggle Kernels**, and **Local Workspace (`uv`)**

## 1. Environment Setup & Path Resolution
ตั้งค่าสภาพแวดล้อมอัตโนมัติ (Local / Kaggle / Colab), ล็อก `SEED = 42` และกำหนดโฟลเดอร์ `data/`, `models/`, `predictions/`, `assets/` ตามโครงสร้าง `ml00x`

In [ ]:
import os
import sys
import warnings
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

for pkg in ["lightgbm", "xgboost", "catboost", "optuna"]:
    try:
        __import__(pkg)
    except ImportError:
        if "google.colab" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
            os.system(f"pip install -q {pkg}")

import lightgbm as lgb
import xgboost as xgb
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path.cwd()
if (BASE_DIR / "project" / "task2").exists():
    TASK_DIR = BASE_DIR / "project" / "task2"
elif (BASE_DIR / "task2").exists():
    TASK_DIR = BASE_DIR / "task2"
else:
    TASK_DIR = BASE_DIR

DATA_DIR = TASK_DIR / "data"
MODELS_DIR = TASK_DIR / "models"
PREDICTIONS_DIR = TASK_DIR / "predictions"
ASSETS_DIR = TASK_DIR / "assets"

for d in [DATA_DIR, MODELS_DIR, PREDICTIONS_DIR, ASSETS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.sans-serif"] = ["Sarabun", "TH Sarabun New", "DejaVu Sans", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

SEGMENT_NAMES = {0: "Casual (0)", 1: "Grinder (1)", 2: "Social (2)", 3: "Whale (3)"}
print(f"Task 2 Directory: {TASK_DIR}")

## 2. Data Loading, Schema Inspection & Noise Feature Filtering
จากการตรวจสอบ `task2/data/train.csv` ($104,000$ แถว $\times 46$ คอลัมน์) และ `datacard.txt`:
* **Target (`segment`):** ไม่มีค่าว่าง (`0=Casual`: 40,934 [39.36%], `1=Grinder`: 26,034 [25.03%], `2=Social`: 21,014 [20.21%], `3=Whale`: 16,018 [15.40%])
* **Explicit Noise Columns (จาก `datacard.txt` ข้อ 32–34):** คอลัมน์ `random_metric_1`, `random_metric_2`, `random_metric_3` ถูกระบุใน Data Card อย่างชัดเจนว่าเป็น *"Random uniform/normal noise - no correlation with segments"* การตัด 3 คอลัมน์นี้ออกช่วยลดการแตกกิ่งที่ผิดพลาด (Overfitting) ของ Tree Models ทันที
* **Categorical Features (10 คอลัมน์):** มีทั้งแบบจัดลำดับได้ (Ordinal: `engagement_level`, `loyalty_tier`, `skill_tier`, `account_status`) และแบบกลุ่มทั่วไป (Nominal: `region`, `platform`, `device_type`, `payment_method`, `language`, `player_type_tag`)

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"Train Shape : {train_df.shape}")
print(f"Test Shape  : {test_df.shape}")
print("\nSegment Distribution:")
for k, v in train_df["segment"].value_counts().sort_index().items():
    print(f"  Class {k} ({SEGMENT_NAMES[k]}): {v:,} ({v/len(train_df)*100:.2f}%)")

NOISE_COLS = ["random_metric_1", "random_metric_2", "random_metric_3"]
CAT_COLS = [
    "region", "platform", "device_type", "payment_method", "language",
    "account_status", "player_type_tag", "engagement_level", "loyalty_tier", "skill_tier"
]
NUM_COLS = [c for c in test_df.columns if c not in ["player_id"] + CAT_COLS + NOISE_COLS]
print(f"\nClean Numerical Features: {len(NUM_COLS)} | Categorical Features: {len(CAT_COLS)} | Dropped Noise: {NOISE_COLS}")

## 3. Exploratory Data Analysis (EDA) & Segment Archetype Visualization
สร้างกราฟเปรียบเทียบสัดส่วนผู้เล่นทั้ง 4 กลุ่ม และโปรไฟล์พฤติกรรมเฉลี่ยของแต่ละ Segment (Playtime, Spending, Ranked Participation, Social Activity) บันทึกลงใน `task2/assets/`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

seg_counts = train_df["segment"].value_counts().sort_index()
colors = ["#3B82F6", "#EF4444", "#10B981", "#F59E0B"]
bars = axes[0].bar([SEGMENT_NAMES[i] for i in seg_counts.index], seg_counts.values, color=colors, width=0.58)
axes[0].set_title("Task 2: Player Segment Distribution (N = 104,000)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Number of Players")
axes[0].grid(axis="y", alpha=0.3)
for b in bars:
    h = b.get_height()
    axes[0].text(b.get_x() + b.get_width()/2, h + 800, f"{h:,}\n({h/len(train_df)*100:.1f}%)", ha="center", fontweight="bold")

key_metrics = ["total_spending_thb", "ranked_participation_rate", "chat_activity_score", "play_frequency"]
norm_profile = train_df.groupby("segment")[key_metrics].mean()
norm_profile = (norm_profile - norm_profile.min()) / (norm_profile.max() - norm_profile.min() + 1e-8)
norm_profile.index = [SEGMENT_NAMES[i] for i in norm_profile.index]
sns.heatmap(norm_profile, annot=True, fmt=".2f", cmap="YlGnBu", ax=axes[1], cbar=True)
axes[1].set_title("Normalized Behavioral Archetype Heatmap by Segment", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig(ASSETS_DIR / "segment-distribution-and-profile.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Behavioral Ratio & Ordinal Feature Engineering
สร้างฟีเจอร์เชิงพฤติกรรมที่แยกความแตกต่างระหว่าง **Casual**, **Grinder**, **Social**, และ **Whale** อย่างชัดเจน:
* **Monetization Intensity (ดึงคลาส Whale):** `spending_per_play_hour`, `avg_tx_size`, `monthly_to_total_spend_ratio`, `vip_x_spending`
* **Competitive Grinding Intensity (ดึงคลาส Grinder):** `ranked_intensity`, `achievement_velocity`, `tournament_per_age`
* **Social Community Index (ดึงคลาส Social):** `social_composite`, `gifts_per_friend`, `invites_per_friend`
* **Ordinal Encoding & Out-of-Fold Frequency Encoding:** แปลงตัวแปรจัดลำดับ (`loyalty_tier`, `skill_tier`, `engagement_level`) เป็นสเกลตัวเลขที่มีความหมาย

In [ ]:
ORDINAL_MAPS = {
    "account_status": {"Dormant": 0, "Active": 1},
    "engagement_level": {"Low": 0, "Mid": 1, "High": 2, "Veteran": 3},
    "loyalty_tier": {"Bronze": 0, "Silver": 1, "Gold": 2, "Platinum": 3, "Diamond": 4},
    "skill_tier": {"Bronze": 0, "Silver": 1, "Gold": 2, "Platinum": 3, "Diamond": 4},
}

NOMINAL_COLS = ["region", "platform", "device_type", "payment_method", "language", "player_type_tag"]

def engineer_task2_features(df: pd.DataFrame, freq_maps: dict = None):
    out = df[NUM_COLS].copy()
    eps = 1e-5
    
    # Missing count
    out["missing_count"] = df[NUM_COLS + CAT_COLS].isna().sum(axis=1)
    
    # 1. Whale / Monetization Ratios
    out["spending_per_hour"] = out["total_spending_thb"] / (out["total_playtime_hours"] + 1.0)
    out["avg_tx_size_est"] = out["avg_monthly_spending"] / (out["spending_frequency"] + 0.1)
    out["monthly_to_lifetime_spend"] = (out["avg_monthly_spending"] * 12.0) / (out["total_spending_thb"] + 1.0)
    out["log_total_spending"] = np.log1p(out["total_spending_thb"].clip(lower=0))
    out["log_monthly_spending"] = np.log1p(out["avg_monthly_spending"].clip(lower=0))
    out["rare_items_per_collection"] = out["rare_items_count"] / (out["collection_progress"] + 1.0)
    
    # 2. Competitive Grinder Ratios
    out["weekly_play_minutes"] = out["play_frequency"] * out["avg_session_duration"]
    out["ranked_win_product"] = out["ranked_participation_rate"] * out["win_rate_ranked"]
    out["competitive_index"] = out["competitive_rank"] * out["ranked_participation_rate"]
    out["playtime_per_day_age"] = out["total_playtime_hours"] / (out["account_age_days"] + 1.0)
    
    # 3. Social Ratios
    out["gifts_per_friend"] = out["gifts_sent_received"] / (out["friend_count"] + 1.0)
    out["invites_per_friend"] = out["friend_invites_sent"] / (out["friend_count"] + 1.0)
    out["social_engagement_product"] = out["team_play_percentage"] * out["chat_activity_score"]
    
    # 4. Preference Dominance
    pref_sum = (
        out["item_type_preference_cosmetic"].fillna(0)
        + out["item_type_preference_performance"].fillna(0)
        + out["item_type_preference_social"].fillna(0) + eps
    )
    out["cosmetic_pref_share"] = out["item_type_preference_cosmetic"] / pref_sum
    out["performance_pref_share"] = out["item_type_preference_performance"] / pref_sum
    out["social_pref_share"] = out["item_type_preference_social"] / pref_sum
    
    # 5. Ordinal Encodings
    for col, mapping in ORDINAL_MAPS.items():
        out[f"{col}_ord"] = df[col].map(mapping).fillna(-1).astype(float)
        
    # 6. One-hot / Categorical features
    for col in NOMINAL_COLS:
        if freq_maps and col in freq_maps:
            out[f"{col}_freq"] = df[col].map(freq_maps[col]).fillna(0.0)
        out[col] = df[col].fillna("MISSING").astype("category")
        
    return out

freq_maps = {col: train_df[col].value_counts(normalize=True).to_dict() for col in NOMINAL_COLS}
X_train_fe = engineer_task2_features(train_df, freq_maps)
X_test_fe = engineer_task2_features(test_df, freq_maps)
y_train = train_df["segment"].astype(int).values

# Prepare numerical-only one-hot version for XGBoost strictly from train schema (Rule 2 compliance: never touch test during schema fit)
X_train_xgb = pd.get_dummies(X_train_fe, columns=NOMINAL_COLS, dummy_na=False)
X_test_xgb = pd.get_dummies(X_test_fe, columns=NOMINAL_COLS, dummy_na=False)
X_test_xgb = X_test_xgb.reindex(columns=X_train_xgb.columns, fill_value=0)

print(f"Engineered Feature Matrix — LGBM: {X_train_fe.shape}, XGB One-Hot: {X_train_xgb.shape}")

## 5. Stratified 5-Fold Training & Class-Prior Multiplier Tuning for Macro $F_1$
ฝึกสอนโมเดล Multi-class GBDT (LightGBM + XGBoost) และใช้เทคนิค **Post-Processing Class Multiplier Optimization (Logit Adjustment)** เพื่อปรับน้ำหนักความน่าจะเป็นของทั้ง 4 คลาส ($w_0, w_1, w_2, w_3$) บน Out-of-Fold ให้ได้ค่า **Macro $F_1$** สูงที่สุด

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
NUM_CLASSES = 4

oof_lgb = np.zeros((len(train_df), NUM_CLASSES))
test_lgb = np.zeros((len(test_df), NUM_CLASSES))

oof_xgb = np.zeros((len(train_df), NUM_CLASSES))
test_xgb = np.zeros((len(test_df), NUM_CLASSES))

feature_importances = np.zeros(X_train_fe.shape[1])

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train_fe, y_train), 1):
    # LightGBM
    model_lgb = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=NUM_CLASSES,
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=40,
        reg_alpha=1.0,
        reg_lambda=2.0,
        random_state=SEED + fold,
        n_jobs=-1,
        verbose=-1
    )
    model_lgb.fit(
        X_train_fe.iloc[trn_idx], y_train[trn_idx],
        eval_set=[(X_train_fe.iloc[val_idx], y_train[val_idx])],
        callbacks=[lgb.early_stopping(stopping_rounds=80, verbose=False)]
    )
    oof_lgb[val_idx] = model_lgb.predict_proba(X_train_fe.iloc[val_idx])
    test_lgb += model_lgb.predict_proba(X_test_fe) / skf.n_splits
    feature_importances += model_lgb.feature_importances_ / skf.n_splits
    
    # XGBoost
    model_xgb = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=NUM_CLASSES,
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=2.0,
        eval_metric="mlogloss",
        early_stopping_rounds=80,
        random_state=SEED + fold,
        n_jobs=-1,
        tree_method="hist"
    )
    model_xgb.fit(
        X_train_xgb.iloc[trn_idx], y_train[trn_idx],
        eval_set=[(X_train_xgb.iloc[val_idx], y_train[val_idx])],
        verbose=False
    )
    oof_xgb[val_idx] = model_xgb.predict_proba(X_train_xgb.iloc[val_idx])
    test_xgb += model_xgb.predict_proba(X_test_xgb) / skf.n_splits
    
    f1_l = f1_score(y_train[val_idx], oof_lgb[val_idx].argmax(axis=1), average="macro")
    f1_x = f1_score(y_train[val_idx], oof_xgb[val_idx].argmax(axis=1), average="macro")
    print(f"Fold {fold}/5 | LGBM Macro F1: {f1_l:.4f} | XGB Macro F1: {f1_x:.4f}")

oof_ens = 0.55 * oof_lgb + 0.45 * oof_xgb
test_ens = 0.55 * test_lgb + 0.45 * test_xgb

# Rule 2 Allowed Enhancement: Combine Full Train + Validation sets for final refit after CV validation
full_lgb_t2 = lgb.LGBMClassifier(
    objective="multiclass", num_class=NUM_CLASSES, n_estimators=800,
    learning_rate=0.03, num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    min_child_samples=40, reg_alpha=1.0, reg_lambda=2.0, random_state=SEED, n_jobs=-1, verbose=-1
)
full_lgb_t2.fit(X_train_fe, y_train)
test_ens = 0.80 * test_ens + 0.20 * full_lgb_t2.predict_proba(X_test_fe)

# Coordinate ascent optimization for class probability multipliers to maximize Macro F1
best_weights = np.ones(NUM_CLASSES)
best_macro_f1 = f1_score(y_train, oof_ens.argmax(axis=1), average="macro")

for _ in range(3):
    for c in range(NUM_CLASSES):
        for w in np.linspace(0.70, 1.40, 71):
            cand_w = best_weights.copy()
            cand_w[c] = w
            preds = (oof_ens * cand_w).argmax(axis=1)
            score = f1_score(y_train, preds, average="macro")
            if score > best_macro_f1:
                best_macro_f1 = score
                best_weights = cand_w

print(f"\nOOF Macro F1 (LGBM)               : {f1_score(y_train, oof_lgb.argmax(axis=1), average='macro'):.4f}")
print(f"OOF Macro F1 (XGBoost)            : {f1_score(y_train, oof_xgb.argmax(axis=1), average='macro'):.4f}")
print(f"OOF Macro F1 (Ensemble Raw)       : {f1_score(y_train, oof_ens.argmax(axis=1), average='macro'):.4f}")
print(f"OOF Macro F1 (Logit-Tuned Weights): {best_macro_f1:.4f} | Weights={best_weights.round(3)}")

## 6. Evaluation Diagnostics, Confusion Matrix & Feature Importance (`task2/assets/`)

In [ ]:
oof_pred_final = (oof_ens * best_weights).argmax(axis=1)
print(classification_report(y_train, oof_pred_final, target_names=[SEGMENT_NAMES[i] for i in range(4)], digits=4))

bench_t2 = pd.DataFrame([
    {"model": "LightGBM (5-Fold)", "macro_f1": f1_score(y_train, oof_lgb.argmax(axis=1), average="macro")},
    {"model": "XGBoost (5-Fold)", "macro_f1": f1_score(y_train, oof_xgb.argmax(axis=1), average="macro")},
    {"model": "Ensemble (LGBM + XGB)", "macro_f1": f1_score(y_train, oof_ens.argmax(axis=1), average="macro")},
    {"model": "Ensemble + Class Multiplier Tuning", "macro_f1": best_macro_f1},
])
bench_t2.to_csv(ASSETS_DIR / "model-benchmark.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
cm = confusion_matrix(y_train, oof_pred_final)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=[SEGMENT_NAMES[i] for i in range(4)],
            yticklabels=[SEGMENT_NAMES[i] for i in range(4)], ax=axes[0])
axes[0].set_title(f"Task 2 OOF Confusion Matrix (Macro F1 = {best_macro_f1:.4f})", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Predicted Segment")
axes[0].set_ylabel("Actual Segment")

fi_df = pd.DataFrame({"feature": X_train_fe.columns, "importance": feature_importances}).sort_values("importance", ascending=False).head(15)
axes[1].barh(fi_df["feature"][::-1], fi_df["importance"][::-1], color="#2563EB")
axes[1].set_title("Task 2: Top 15 Features for Player Segmentation", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Average Split Importance (LightGBM 5-Fold)")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(ASSETS_DIR / "confusion-matrix-and-feature-importance.png", dpi=300, bbox_inches="tight")
plt.show()

# Save OOF & Test predictions
oof_out = pd.DataFrame({
    "player_id": train_df["player_id"],
    "segment_true": y_train,
    "segment_pred": oof_pred_final,
})
oof_out.to_csv(PREDICTIONS_DIR / "oof_predictions.csv", index=False)

test_pred_final = (test_ens * best_weights).argmax(axis=1)
test_out = pd.DataFrame({
    "player_id": test_df["player_id"],
    "task2": test_pred_final.astype(int),
})
for c in range(NUM_CLASSES):
    test_out[f"prob_class_{c}"] = test_ens[:, c]
test_out.to_csv(PREDICTIONS_DIR / "test_predictions.csv", index=False)

print(f"Saved Test predictions: {PREDICTIONS_DIR / 'test_predictions.csv'} ({len(test_out):,} rows)")
print("Predicted Test Segment Counts:\n", test_out["task2"].value_counts().sort_index())